# Regularization - Ridge, Lasso & Elastic Net

## 🎯 Goal
Compare regularization techniques on a dataset with many features (some useful, some noise)

## 📚 What You'll Learn
- Why regularization is needed (overfitting)
- Ridge (L2) - shrinks all coefficients
- Lasso (L1) - sets coefficients to zero (feature selection)
- Elastic Net - combines both
- How to tune hyperparameters with cross-validation

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)
sns.set_style('whitegrid')

print('✅ Libraries imported!')

## Step 2: Create Dataset with Noise Features

We'll create a dataset where only **5 out of 50 features** are actually useful. Regularization should detect this!

In [ ]:
# Generate dataset
n_samples = 200
n_features = 50  # Many features
n_useful = 5    # Only 5 are useful

X = np.random.randn(n_samples, n_features)

# Only first 5 features are useful
true_coefs = np.zeros(n_features)
true_coefs[:n_useful] = [3.5, -2.5, 4.0, -1.5, 5.0]

y = X @ true_coefs + np.random.randn(n_samples) * 1.0  # With noise

print(f'📊 Dataset Created:')
print(f'   Samples: {n_samples}')
print(f'   Total features: {n_features}')
print(f'   Useful features: {n_useful}')
print(f'   Noise features: {n_features - n_useful}')
print(f'\n💡 True coefficients (first 10):')
for i in range(10):
    label = '✅ USEFUL' if i < n_useful else '❌ NOISE'
    print(f'   Feature {i}: {true_coefs[i]:6.2f} {label}')

# Convert to DataFrame
feature_names = [f'feat_{i}' for i in range(n_features)]
X_df = pd.DataFrame(X, columns=feature_names)

## Step 3: Split and Scale

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

# Scale features (CRITICAL for regularization!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'📐 Train: {X_train_scaled.shape}')
print(f'📐 Test:  {X_test_scaled.shape}')
print('\n✅ Always scale before regularization!')

## Step 4: Train Models - Compare All Methods

In [ ]:
models = {
    'Linear (no reg)': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results = {}
all_coefs = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    n_zero = np.sum(np.abs(model.coef_) < 1e-5)
    
    results[name] = {
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Gap': train_r2 - test_r2,
        'Zero Coefs': n_zero
    }
    
    all_coefs[name] = model.coef_

results_df = pd.DataFrame(results).T
print('📊 Model Comparison:')
print('=' * 60)
print(results_df.round(3))

print('\n💡 Look for:')
print('   - Smallest GAP between Train and Test R² → less overfit')
print('   - Many Zero Coefs → automatic feature selection (Lasso!)')

## Step 5: Visualize Coefficients

See how each method handles the coefficients!

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, (name, coefs) in zip(axes, all_coefs.items()):
    colors = ['green' if i < n_useful else 'red' for i in range(n_features)]
    ax.bar(range(n_features), coefs, color=colors, alpha=0.7, edgecolor='black')
    
    # Mark useful feature region
    ax.axvspan(-0.5, n_useful-0.5, alpha=0.1, color='green', label='Useful features')
    
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature Index')
    ax.set_ylabel('Coefficient')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Coefficient Comparison (Green=Useful, Red=Noise)', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 Observations:')
print('   - Linear: Many noisy non-zero coefficients (overfit)')
print('   - Ridge: All coefs shrunk but none are zero')
print('   - Lasso: Many coefs are exactly zero (selected!)')
print('   - Elastic Net: Mix of both')

## Step 6: Effect of Alpha on Ridge

In [ ]:
alphas = np.logspace(-3, 3, 50)
ridge_coefs = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)

ridge_coefs = np.array(ridge_coefs)

fig, ax = plt.subplots(figsize=(12, 7))
for i in range(n_features):
    color = 'green' if i < n_useful else 'red'
    alpha_val = 0.8 if i < n_useful else 0.3
    lw = 2 if i < n_useful else 0.8
    ax.plot(alphas, ridge_coefs[:, i], color=color, alpha=alpha_val, linewidth=lw)

ax.set_xscale('log')
ax.set_xlabel('Alpha (Regularization Strength)', fontsize=12)
ax.set_ylabel('Coefficient Value', fontsize=12)
ax.set_title('Ridge: Coefficients Shrink as Alpha Increases', 
             fontsize=14, fontweight='bold')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='green', lw=2, label='Useful features'),
    Line2D([0], [0], color='red', lw=1, label='Noise features')
]
ax.legend(handles=legend_elements, fontsize=11)
plt.tight_layout()
plt.show()

## Step 7: Effect of Alpha on Lasso (Feature Selection!)

In [ ]:
alphas = np.logspace(-3, 1, 50)
lasso_coefs = []
n_features_selected = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)
    n_features_selected.append(np.sum(np.abs(lasso.coef_) > 1e-5))

lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Coefficients
for i in range(n_features):
    color = 'green' if i < n_useful else 'red'
    alpha_val = 0.8 if i < n_useful else 0.3
    lw = 2 if i < n_useful else 0.8
    axes[0].plot(alphas, lasso_coefs[:, i], color=color, alpha=alpha_val, linewidth=lw)

axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('Coefficient', fontsize=12)
axes[0].set_title('Lasso: Coefficients (some go to zero!)', 
                  fontsize=13, fontweight='bold')
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].grid(True, alpha=0.3)

# Plot 2: Number of features selected
axes[1].plot(alphas, n_features_selected, linewidth=3, color='blue')
axes[1].axhline(y=n_useful, color='green', linestyle='--', 
                label=f'True useful features = {n_useful}')
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha', fontsize=12)
axes[1].set_ylabel('Features Selected', fontsize=12)
axes[1].set_title('Lasso: Feature Count Decreases with Alpha', 
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Auto-tune with Cross-Validation

In [ ]:
# Define alpha range
alphas = np.logspace(-4, 2, 100)

# Auto-tune Ridge
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

# Auto-tune Lasso
lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

# Auto-tune Elastic Net
elastic_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95],
    alphas=alphas, cv=5, max_iter=10000, random_state=42
)
elastic_cv.fit(X_train_scaled, y_train)

print('🎯 Auto-Tuned Hyperparameters:')
print('=' * 50)
print(f'\nRidge:')
print(f'   Best alpha: {ridge_cv.alpha_:.4f}')
print(f'\nLasso:')
print(f'   Best alpha: {lasso_cv.alpha_:.4f}')
n_selected = np.sum(np.abs(lasso_cv.coef_) > 1e-5)
print(f'   Features selected: {n_selected}/{n_features}')
print(f'\nElastic Net:')
print(f'   Best alpha: {elastic_cv.alpha_:.4f}')
print(f'   Best l1_ratio: {elastic_cv.l1_ratio_}')

## Step 9: Compare Tuned Models

In [ ]:
tuned_models = {
    'Linear': LinearRegression(),
    'Ridge (tuned)': ridge_cv,
    'Lasso (tuned)': lasso_cv,
    'Elastic Net (tuned)': elastic_cv
}

# Re-fit Linear for comparison
tuned_models['Linear'].fit(X_train_scaled, y_train)

comparison = []
for name, model in tuned_models.items():
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    
    n_zero = np.sum(np.abs(model.coef_) < 1e-5)
    
    comparison.append({
        'Model': name,
        'Train R²': r2_score(y_train, train_pred),
        'Test R²': r2_score(y_test, test_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Zero Coefs': n_zero,
        'Features Used': n_features - n_zero
    })

comp_df = pd.DataFrame(comparison)
print('🏆 Tuned Models Comparison:')
print('=' * 70)
print(comp_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# R² comparison
x_pos = np.arange(len(comp_df))
width = 0.35
axes[0].bar(x_pos - width/2, comp_df['Train R²'], width, label='Train R²', alpha=0.7)
axes[0].bar(x_pos + width/2, comp_df['Test R²'], width, label='Test R²', alpha=0.7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comp_df['Model'], rotation=20)
axes[0].set_ylabel('R² Score')
axes[0].set_title('Train vs Test R² (smaller gap = better)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Features used
axes[1].bar(comp_df['Model'], comp_df['Features Used'], color='purple', alpha=0.7)
axes[1].axhline(y=n_useful, color='green', linestyle='--', 
                label=f'True useful features = {n_useful}')
axes[1].set_ylabel('Features Used')
axes[1].set_title('Features Used by Each Model')
axes[1].set_xticklabels(comp_df['Model'], rotation=20)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Lasso's Selected Features

In [ ]:
# Show what Lasso selected
lasso_selected = np.where(np.abs(lasso_cv.coef_) > 1e-5)[0]

print('🎯 Features Lasso Selected:')
print('=' * 50)
for idx in lasso_selected:
    is_useful = '✅ TRUE positive' if idx < n_useful else '❌ FALSE positive'
    print(f'   Feature {idx}: coef={lasso_cv.coef_[idx]:.3f}  {is_useful}')

print(f'\n📊 Lasso Performance:')
print(f'   Total selected: {len(lasso_selected)}/{n_features}')
true_positives = np.sum([1 for i in lasso_selected if i < n_useful])
print(f'   True useful features found: {true_positives}/{n_useful}')
print(f'   False positives: {len(lasso_selected) - true_positives}')

if true_positives == n_useful:
    print('\n   🎉 Lasso correctly identified ALL useful features!')

## 🎓 Summary

### What We Demonstrated:
1. ✅ Created a dataset with many noise features
2. ✅ Showed how Linear Regression overfits (big train-test gap)
3. ✅ Ridge shrinks all coefficients (handles multicollinearity)
4. ✅ Lasso eliminates features (automatic selection!)
5. ✅ Elastic Net combines both approaches
6. ✅ Used cross-validation to auto-tune hyperparameters
7. ✅ Verified Lasso correctly identified useful features

### Key Takeaways:
- **Always scale features** before regularization
- **Use cross-validation** to tune alpha
- **Ridge** for stable predictions with all features
- **Lasso** for feature selection
- **Elastic Net** for correlated features + selection
- **Smaller train-test gap** = better generalization

### When to Use What:

| Situation | Use |
|-----------|-----|
| All features matter | Ridge |
| Want feature selection | Lasso |
| Correlated features | Elastic Net |
| Small dataset | Ridge |
| Large dataset, many features | Lasso/Elastic Net |

**Happy Modeling! 🎯✨**